In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from pathlib import Path

notebook_path = "/u/skarmakar1/version_check/llm_steering-main/sk"
sys.path.append(notebook_path)

In [3]:
import torch
import numpy as np

from inversion_utils import *
import pickle
from sklearn.model_selection import train_test_split

In [4]:
SEED = 0
# SEED = 1

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)

torch.backends.cudnn.benchmark = True 
torch.backends.cuda.matmul.allow_tf32 = True

LLM = namedtuple('LLM', ['language_model', 'tokenizer', 'processor', 'name', 'model_type'])

In [ ]:
import random
import json

random.seed(SEED)

In [6]:
# model_type = 'llama'
# MODEL_VERSION='3.1'
# MODEL_SIZE='8B'

# model_type = 'gemma'
# MODEL_VERSION='2'
# # MODEL_VERSION='3'
# # MODEL_SIZE='1B'
# MODEL_SIZE='9B'
# # MODEL_SIZE='12B'

model_type = 'qwen'
MODEL_VERSION='3'
# MODEL_VERSION='3t'
# MODEL_SIZE='4B'
# MODEL_SIZE='8B'
# MODEL_SIZE='14B'
MODEL_SIZE='32B'
# MODEL_SIZE='30B'

llm = select_llm(model_type, MODEL_VERSION=MODEL_VERSION, MODEL_SIZE=MODEL_SIZE)

Loading checkpoint shards:   0%|          | 0/17 [00:00<?, ?it/s]

In [7]:
hidden_layers = list(range(-llm.language_model.config.num_hidden_layers+1, 0))
print(hidden_layers)

[-63, -62, -61, -60, -59, -58, -57, -56, -55, -54, -53, -52, -51, -50, -49, -48, -47, -46, -45, -44, -43, -42, -41, -40, -39, -38, -37, -36, -35, -34, -33, -32, -31, -30, -29, -28, -27, -26, -25, -24, -23, -22, -21, -20, -19, -18, -17, -16, -15, -14, -13, -12, -11, -10, -9, -8, -7, -6, -5, -4, -3, -2, -1]


Evalute

In [ ]:
# data_file = f"../all_gitignore/eval_json_auto/inv/combined.json"
# data_file = f"../all_gitignore/eval_json_auto/inv/separate.json"
data_file = f"../all_gitignore/eval_json_auto/inv/orig.json"

with open(data_file, 'r', encoding='utf-8') as file:
    data = json.load(file)

In [ ]:
base_prompt = """You are an expert semantic evaluator. Your task is to judge how well a model is able to generate a steering vector of a concept for a language model.

I will provide you with:
1. A TARGET CONCEPT (e.g., "Happy", "Large", "True").
2. A USER PROMPT asked to the model.
3. The STEERED RESPONSE (generated by adding the learned steering vector to the activations of the language model).

### YOUR GOAL
Determine if the **Steered Response** successfully embodies the **Target Concept**. Evaluate if the **Steered Response** is successfully steered towards the given **Target Concept**.

### SCORING CRITERIA (0-3)
- **3 (Perfect Prediction):** The Steered Response perfectly embodies the Target Concept. (e.g., If Concept is "Happy", the response is clearly shows happiness).
- **2 (Good Prediction):** The Steered Response clearly leans towards the given concept, though it could be more intense or explicit.
- **1 (Unclear):** The Predicted Response does not show any signs of the **Target Concept**.
- **0 (Failure):** The Steered Response embodies the opposite of the *original* Target Concept, steering failed completely, or is incoherent.

### OUTPUT
You will return your answer in the following format and nothing else.

Score: <a single score (0-3)>
Justification: <a one to two line brief justification>

### INPUT DATA
- **Target Concept**: {target_concept}
- **User Prompt**: {prompt}
- **Steered Response**: {response}"""

In [10]:
# 30b - thinking ~15 min, non thinking ~5 min
# 14b - thinking ~40 sec, non thinking ~22 sec
# 8b - thinking ~40 sec, non thinking ~20 sec

In [11]:
# messages = [
#     {"role": "user", "content": "explain how a clock works."}
# ]
# text = llm.tokenizer.apply_chat_template(
#     messages,
#     tokenize=False,
#     add_generation_prompt=True,
#     enable_thinking=True, # Switches between thinking and non-thinking modes. Default is True.
#     # enable_thinking=False,
# )
# model_inputs = llm.tokenizer([text], return_tensors="pt").to(llm.language_model.device)

# # conduct text completion
# generated_ids = llm.language_model.generate(
#     **model_inputs,
#     max_new_tokens=32768
# )
# output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# # parsing thinking content
# try:
#     # rindex finding 151668 (</think>)
#     index = len(output_ids) - output_ids[::-1].index(151668)
# except ValueError:
#     index = 0

# thinking_content = llm.tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
# content = llm.tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

# print("thinking content:", thinking_content)
# print("content:", content)

In [12]:
def evaluate(llm, prompt):
    # needs qwen
    
    # prompt = base_prompt.format(target_concept=t, prompt=p, response=r)

    messages = [
        {"role": "user", "content": prompt}
    ]
    text = llm.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True, # Switches between thinking and non-thinking modes. Default is True.
        # enable_thinking=False,
    )
    model_inputs = llm.tokenizer([text], return_tensors="pt").to(llm.language_model.device)

    # conduct text completion
    generated_ids = llm.language_model.generate(
        **model_inputs,
        max_new_tokens=32768,
        # ---------------------
        do_sample=True,
        temperature=0.6,
        top_k=20,
        top_p=0.95,
        # ---------------------
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

    # parsing thinking content
    try:
        # rindex finding 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    thinking_content = llm.tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    content = llm.tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

    # print("thinking content:", thinking_content)
    # print("content:", content)

    return content

In [ ]:
result = {}

for i in tqdm(data):
    target_concept = data[i]["Antonym"]
    prompt = data[i]["User Prompt"]
    response = data[i]["Predicted Antonym Response"]

    eval_prompt = base_prompt.format(target_concept=target_concept, prompt=prompt, response=response)

    result[i] = {
        "content": evaluate(llm, eval_prompt),
        "target_concept": target_concept,
        "prompt": prompt,
        "response": response,
    }

    # break

In [ ]:
# filename = f"../all_gitignore/eval_json_auto/inv/combined_result.json"
# filename = f"../all_gitignore/eval_json_auto/inv/separate_result.json"
filename = f"../all_gitignore/eval_json_auto/inv/orig_result.json"

with open(filename, 'w') as f:
    json.dump(result, f, indent=4)

Parser

In [1]:
import json

In [2]:
# filename = f"../all_gitignore/eval_json_auto/inv/combined_result.json"
# filename = f"../all_gitignore/eval_json_auto/inv/separate_result.json"
filename = f"../all_gitignore/eval_json_auto/inv/orig_result.json"

with open(filename, 'r', encoding='utf-8') as file:
    result = json.load(file)

In [3]:
# print(result)

In [4]:
def score_parser(response):
    score_string = "Score: "
    try:
        score_end_index = response.index(score_string) + len(score_string)
    except ValueError:
        print(f"'{score_string}' not found. Got this\n{response}")

    just_string = "Justification: "
    try:
        just_start_index = response.index(just_string)
    except ValueError:
        print(f"'{just_string}' not found. Got this\n{response}")
    
    score = response[score_end_index:just_start_index-1]

    # print(score)
    return int(score)

In [5]:
scores = []

for i in result:
    # print(result[i])
    scores.append(score_parser(result[i]["content"]))

In [6]:
print(sum(scores) / len(scores))

1.7783783783783784


In [ ]:
# combined: 1.44
# separate: 1.56
# orig: 1.77